#### 5. Deliberately insert a row with an extra column and observe Delta's schema enforcement rejecting it; then re-insert using mergeSchema and confirm schema evolution succeeded.

In [0]:
insert into dev.demo.sample_products values (12, 'Voltas AC', 'Electronics', 10000.00, 'India')

In [0]:

merge with schema evolution into dev.demo.sample_products t
using (
    select 12 as product_id, 'Voltas AC' as name, 'Electronics' as category, 10000.00 as price, 'India' as country
    union all
    select 11, 'Samsung TV', 'Electronics', 20000.00, 'India'
) s
on t.product_id = s.product_id
when matched then update set t.name = s.name, t.category = s.category, t.price = s.price
when not matched then insert (product_id, name, category, price) values (s.product_id, s.name, s.category, s.price);

select * from dev.demo.sample_products

#### 6. Use time travel (VERSION AS OF and TIMESTAMP AS OF) to reconstruct the table as it looked before a **simulated** bad update, then write the RESTORE command that would fix it.

In [0]:
describe history dev.demo.sample_products

In [0]:
select * from dev.demo.sample_products version as of 6

In [0]:
restore table dev.demo.sample_products to version as of 6

In [0]:
select * from dev.demo.sample_products

#### 7. Write a short explanation, aimed at a non-technical stakeholder, of why ACID transactions matter when multiple pipelines write to the same table concurrently.

So, you might think why this ACID terminology is used by the Developers and they try to tech you with the full form. But, i'll teach you each and everything about ACID, and why it matters when multiple pipelines are writing on to the same data or tables simultaneously along with the real world example to help to understsnd better. This method ensures the overall Integrity of the Enterprise Systems from a small scale to large scale.

Lets break down this:
1. Atomicity: 
It refers to either the change will happen at one transaction or nothing will happen at all if fail. There is no half or semi done transaction in Databases. This helps in a rapid changing environment just to ensure there is only one write at a moment and there is either success or failure(nothing changed).

Example:
 What if you are working with Banking Database and there are around a 1 million users are using the banking system a moment. Now, what is 5 person sending money to a single user and at the same time that single user is also sending money to some other. Now, here comes the Database's Atomicity it will Update the balance of the user in realtime and one at a moment each transaction will add in queue if the one transaction fails then it must not write anything at all. We can performs reading all at once but cannot write at once.

2. Consistency:
Now, what if you are using your banking app from multiple devices like mobile application, desktop application and website of the bank. What if the balance is vary everywhere? ..... Now, this might be the case with the databases too. 

Example:
 Lets continue with the example of banking transaction, suppose if customer A sends money to customer B. Now what if there is a tansaction added in the banking application of the customer A that he/she sends money and their balance is deducted. But, customer B's banking app showing that their balance is same nothing came in or credited. Then where the money goes??? .... This is one of the cases where we need consistency of the data in out database.

3. Isolation:
This is the very important concept if we are working in a concurrent or multi pipelined environment because no one can access the data without the appropriate permissions or authority. All the logics and data must be Isolated so to help business keep their data safe. In a multi-tenant system where the same hardware is shared among all the users, it is very important for a Database to be safe and secure from external environment.

Example:
 Now, suppose you and your friends are having bank accounts in a same bank. What if you deposited some money into your account and you successfully deposited your money and when you checked your balance that amount is not available in your account and deposited into your friends account instead? Now, think what if it reflected to somebody else account then, do you able to recover your money? May be yes or no. So, your account must be isolated form others and do not affected by millions of transactions happens daily into the bank server.

4. Durability:
It is the final concept to achieve the Integrity of any organisation or enterprise. Durability refers to the correctness and maintainability of the system. Let me help you better understand with an example.

Example:
 Let's take the same example of the Bank, Now you have deposited $10000 into your account and what if you didn't made any transaction for over 5 years. So, your balance must be preserved into the bank account even after your death if you didn't cashed anything. 